# EmerKit — séries avec un modèle ouvert (Qwen3-8B) dans Google Colab

Ce notebook installe le dépôt, sert **Qwen3-8B** avec Ollama sur le GPU de Colab (contexte porté à 32 768 jetons), puis lance les séries qui demandent un second modèle : sondes sur outils réels (A5, avec le test de saturation), C1 (Compositio Prompto et générateur LLM + EmerKit), exécution réelle avec un agent (F1), extraction des capacités (A0) et panel de juges.

**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4**. Puis Exécution → **Tout exécuter**. Durée indicative : 2 à 4 heures. Les résultats sont copiés sur votre Google Drive au fur et à mesure (cellule 9), et une archive finale est téléchargée (cellule 10) : téléversez-la ensuite dans la conversation.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Google Drive (sauvegarde) et dépôt EmerKit
from google.colab import drive, files
import os, zipfile, glob
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/emerkit_qwen'; os.makedirs(SAVE, exist_ok=True)
zips = glob.glob('/content/drive/MyDrive/emerkit_github.zip')
if zips:
    path = zips[0]
else:
    print('Choisissez emerkit_github.zip'); path = list(files.upload().keys())[0]
zipfile.ZipFile(path).extractall('/content')
%cd /content/emerkit
!ls

In [ ]:
# 2. Dépendances et données tierces
!pip -q install -e ".[llm]"
!python scripts/fetch_data.py

In [ ]:
# 3. Ollama et Qwen3-8B, avec un contexte de 32 768 jetons (le contexte par défaut d'Ollama tronquerait les longues listes d'outils)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os
os.environ['OLLAMA_NUM_PARALLEL'] = '4'
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen3:8b
open('/content/Modelfile', 'w').write('FROM qwen3:8b\nPARAMETER num_ctx 32768\n')
!ollama create qwen3-8b-32k -f /content/Modelfile
!ollama list

In [ ]:
# 4. Configuration d'EmerKit pour le modèle local (API compatible OpenAI d'Ollama)
os.environ.update(EMERKIT_PROVIDER='openai', EMERKIT_BASE_URL='http://localhost:11434/v1', OPENAI_API_KEY='ollama',
                  EMERKIT_MODEL='qwen3-8b-32k', EMERKIT_NO_THINK='1', EMERKIT_WORKERS='4')
import sys; sys.path.insert(0, '/content/emerkit')
from emerkit.llm import LLMClient
print(LLMClient()('Return only the word ok.', 'Reply with ok.', max_tokens=20))

In [ ]:
# 5. A5 sur outils réels : 500 sondes (requêtes écrites par un autre modèle), puis ajustement sur tous les modèles disponibles
!python scripts/run_probes_real.py probe
!python scripts/run_probes_real.py fit
!python scripts/fit_interference.py
!cp -r results/llm config/fitted_weights*.json $SAVE/ 2>/dev/null; echo sauvegardé

In [ ]:
# 6. C1 : Compositio Prompto et générateur LLM + EmerKit, trois graines
for s in (0, 1, 2):
    !python scripts/run_llm.py c1 {s}
!cp -r results/llm $SAVE/

In [ ]:
# 7. F1 : exécution réelle par un agent Qwen3 via la passerelle MCP, trois graines
for s in (0, 1, 2):
    !python scripts/run_f1.py {s} 20 llm 2>&1 | grep -v Processing | tail -1
!cp -r results/f1 $SAVE/

In [ ]:
# 8. A0 (extraction des capacités) et panel de juges
!python scripts/run_llm.py capabilities 0
!python scripts/run_llm.py judges
!cp -r results/llm $SAVE/

In [ ]:
# 9. Sauvegarde complète sur Google Drive (journal des appels compris)
!cp -r results $SAVE/ ; cp llm_log.jsonl $SAVE/ 2>/dev/null; ls $SAVE

In [ ]:
# 10. Archive à téléverser dans la conversation
!cd /content/emerkit && zip -qr /content/emerkit_qwen_results.zip results/llm results/f1 config/fitted_weights*.json llm_log.jsonl
files.download('/content/emerkit_qwen_results.zip')